# 04. ImageNet-képek sorozatos osztályozása egy könyvtárból

Ez a laboratórium **képfájlokon** fut végig, a [02.](02_dorefanet-imagenet-samples.ipynb) és [03.](03_dorefanet-classification.ipynb) laboratórium CPU/FPGA-modelljét használva. Nem kamerás vagy videós bemutató.

![DoReFaNet topológia](../dorefanet-topology.svg)

A cellákat sorrendben futtassa. Állítsa le a többi FPGA-kernelt. Szükség esetén tartsa aktívan a swapot a nagy súlyfájlok miatt. A telepített csomag nem használt `Xlnk` importjait már el kellett távolítani.

*A fordított példány hardveres validálása még nem történt meg.*

## 1. Fájlok ellenőrzése

Használja a meglévő **imagenet-samples** könyvtárat. A program csak képfájlokat választ ki; más fájlokat és alkönyvtárakat figyelmen kívül hagy.

In [ ]:
import os
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image
from IPython.display import display, clear_output

import qnn
from qnn import Dorefanet, utils

QNN_ROOT = Path(qnn.__file__).resolve().parent
QNN_PARAM_DIR = QNN_ROOT / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "imagenet-classes.pkl").is_file():
    if (NOTEBOOK_DIR.parent / "imagenet-classes.pkl").is_file():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
SAMPLE_DIR = NOTEBOOK_DIR / "imagenet-samples"
LABEL_FILE = NOTEBOOK_DIR / "imagenet-classes.pkl"

required_files = [QNN_PARAM_DIR / name for name in (
    "dorefanet-layers.json", "dorefanet-conv0.npy",
    "dorefanet-fc-normalized.npy", "ilsvrc_pp_mean.npy")]
required_files.append(LABEL_FILE)
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing:\n" + "\n".join(missing))
if not SAMPLE_DIR.is_dir():
    raise FileNotFoundError("Sample folder missing: " + str(SAMPLE_DIR))
image_files = sorted(path for path in SAMPLE_DIR.iterdir()
                     if path.is_file() and path.suffix.lower() in
                     (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("No sample images found in " + str(SAMPLE_DIR))
print("Python package:", QNN_ROOT)
print("Notebook data:", NOTEBOOK_DIR)
print("Sample images:", len(image_files))

## 2. Súlyok és címkék betöltése, a modell inicializálása

Futtassa a következő cellát, és várja meg a **Model ready** üzenetet. Betölti a NumPy-súlyszótárakat és az 1000 ImageNet-címkét, majd inicializálja az FPGA-gyorsítót. A telepített csomag alapértelmezett hálózati beállításait használja.

Minden nagyobb betöltést üzenet jelez. Hiba esetén ne folytassa az inferenciát. Újrafuttatás előtt a korábbi osztályozó felszabadul. A pickle- és NumPy-objektumfájlok a telepített, megbízható csomag részei.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
# Avoid holding old weight dictionaries while reloading this cell.
for variable in ("conv0_weights", "fc_weights"):
    globals().pop(variable, None)
gc.collect()

print("Loading first-convolution weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-conv0.npy"),
                       encoding="latin1", allow_pickle=True).item()
print("Loading fully connected weights; please wait...", flush=True)
fc_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-fc-normalized.npy"),
                    encoding="latin1", allow_pickle=True).item()
for key in ("conv0/W", "conv0/T"):
    if key not in conv0_weights:
        raise KeyError("Missing weight: " + key)
for key in ("fc0/Wn", "fc0/bn", "fc1/Wn", "fc1/bn", "fct/W"):
    if key not in fc_weights:
        raise KeyError("Missing weight: " + key)

with LABEL_FILE.open("rb") as label_stream:
    classes = pickle.load(label_stream, encoding="latin1")
if set(classes.keys()) != set(range(1000)):
    raise ValueError("Expected ImageNet labels indexed from 0 to 999.")
names = {index: classes[index][1].split(",")[0] for index in classes}
synset_to_index = {classes[index][0]: index for index in classes}

print("Initializing the FPGA...", flush=True)
classifier = Dorefanet()
try:
    net = classifier.load_network()
    classifier.init_accelerator()
except BaseException:
    classifier.deinit_accelerator()
    raise
print("Model ready. ImageNet classes:", len(classes))

## 3. Feldolgozófüggvény meghatározása

Ez a cella definiálja az eredeti első CPU-konvolúciót, az FPGA-rétegeket és a végső CPU-rétegeket, de képeket még nem dolgoz fel. A pontszámok a kombinált feldolgozási folyamat eredményei.

In [ ]:
def classify_image(image_path):
    img, img_class = classifier.load_image(str(image_path))
    stage_seconds = {}
    start = time.perf_counter()
    conv0 = utils.conv_layer(img, conv0_weights["conv0/W"], stride=4)
    conv0 = utils.threshold(conv0, conv0_weights["conv0/T"])
    stage_seconds["First convolution + threshold (CPU)"] = time.perf_counter() - start
    out_dim = net["merge4"]["output_dim"]
    out_ch = net["merge4"]["output_channels"]
    conv_output = classifier.get_accel_buffer(out_ch, out_dim)
    conv_input = np.ascontiguousarray(classifier.prepare_buffer(conv0))
    start = time.perf_counter()
    classifier.inference(conv_input, conv_output)
    stage_seconds["Quantized-layer offload (FPGA)"] = time.perf_counter() - start
    conv_output = classifier.postprocess_buffer(conv_output)
    if not np.all(np.isfinite(conv_output)):
        raise ValueError("Non-finite accelerator output. Check model initialization.")
    scale = np.max(conv_output)
    if scale <= 0:
        raise RuntimeError("Accelerator output is all zero. Stop and check initialization before continuing.")
    fc_input = conv_output / scale
    if fc_input.size != fc_weights["fc0/Wn"].shape[0]:
        raise ValueError("Accelerator output size does not match the fully connected weights.")
    start = time.perf_counter()
    fc0_out = utils.fully_connected(fc_input, fc_weights["fc0/Wn"], fc_weights["fc0/bn"])
    fc0_out = utils.quantize(utils.qrelu(fc0_out), 2)
    fc1_out = utils.fully_connected(fc0_out, fc_weights["fc1/Wn"], fc_weights["fc1/bn"])
    fc1_out = utils.qrelu(fc1_out)
    fct_b = np.zeros((fc_weights["fct/W"].shape[1],))
    fct_out = utils.fully_connected(fc1_out, fc_weights["fct/W"], fct_b)
    stage_seconds["Fully connected layers (CPU)"] = time.perf_counter() - start
    if np.asarray(fct_out).shape != (len(classes),) or not np.all(np.isfinite(fct_out)):
        raise ValueError("Expected 1,000 finite classification scores.")
    out = utils.softmax(fct_out)
    topn = utils.get_topn_indexes(out, 5)
    return out, topn, synset_to_index.get(img_class), stage_seconds


## 4. Korlátozott számú képet feldolgozó bemutató

Alapértelmezés szerint legfeljebb **10 képet dolgoz fel, mindegyiket egyszer**, megjelenítve a képet, az ismert osztályt és az öt legnagyobb pontszámú predikciót. A `MAX_IMAGES` értéke tetszőleges pozitív egészre módosítható. A `REPEAT = True` ismételten bejárja a könyvtárat a megadott korlátig.

Korai leállításhoz használja a **Kernel → Interrupt** parancsot, és várja meg a natív hívás visszatérését. A tisztítás normál befejezés, megszakítás vagy Python-hiba esetén is megtörténik. A ciklus leállásáig ne indítson másik FPGA-notebookot. Újrafuttatáshoz ismételje meg a **2. szakaszt**, majd ezt a cellát. Az eredményként számított pontosság kizárólag a feldolgozott, címkézett mintákra vonatkozik.

In [ ]:
MAX_IMAGES = 10  # Positive limit; the demo always stops automatically.
REPEAT = False   # True cycles through the folder, still stopping at MAX_IMAGES.
results = []
try:
    if not isinstance(MAX_IMAGES, int) or isinstance(MAX_IMAGES, bool) or MAX_IMAGES < 1:
        raise ValueError("MAX_IMAGES must be a positive integer.")
    paths = image_files[:MAX_IMAGES] if not REPEAT else [
        image_files[index % len(image_files)] for index in range(MAX_IMAGES)]
    for number, image_path in enumerate(paths, 1):
        print("Processing {}/{}: {}".format(number, len(paths), image_path.name), flush=True)
        started = time.perf_counter()
        scores, topn, expected, stages = classify_image(image_path)
        elapsed = time.perf_counter() - started
        clear_output(wait=True)
        print("Image {}/{}: {}".format(number, len(paths), image_path.name))
        with Image.open(str(image_path)) as opened:
            preview = opened.convert("RGB")
        preview.thumbnail((480, 360))
        display(preview)
        print("Known class:", names[expected] if expected is not None else "unknown")
        for index in topn:
            print("{:25s} {:7.2%}".format(names[int(index)], float(scores[index])))
        print("Processing time: {:.3f} s (image loading + model + scores; excludes display)".format(elapsed))
        results.append(dict(image=image_path.name, expected=expected,
                            top1=int(topn[0]), top5=[int(index) for index in topn], seconds=elapsed))
except KeyboardInterrupt:
    print("Stopped by Kernel > Interrupt.")
finally:
    classifier.deinit_accelerator()
    print("Accelerator released. Completed images:", len(results))
labelled = [row for row in results if row["expected"] is not None]
if labelled:
    print("Top-1 on this processed selection: {:.1%}".format(
        sum(row["top1"] == row["expected"] for row in labelled) / len(labelled)))
    print("Top-5 on this processed selection: {:.1%}".format(
        sum(row["expected"] in row["top5"] for row in labelled) / len(labelled)))
    print("This is a sample/demo score, not full ImageNet validation accuracy.")


## 5. Nagy súlytömbök felszabadítása

A ciklus már felszabadította a gyorsítót. Futtassa ezt a cellát a nagy tömbök eltávolítására. Új indításhoz térjen vissza a 2. szakaszhoz.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
for variable in ("conv0_weights", "fc_weights", "img", "conv0", "conv_input",
                 "conv_output", "fc_input", "fc0_out", "fc1_out", "fct_out"):
    globals().pop(variable, None)
gc.collect()
print("Classifier released and large Python arrays cleared.")

**Validálás:** a Python-folyamat és a hibakezelés off-board tesztelése szimulált gyorsító/natív interfészekkel és az eredeti NumPy-segédfüggvényekkel történt. Az FPGA, a natív Darknet és a predikciók helyessége további PYNQ-hardveres ellenőrzést igényel.